# 🛠️ Active Learning Workshop: Implementing an Inverted Matrix (Jupyter + GitHub Edition)
## 🔍 Workshop Theme
*Readable, correct, and collaboratively reviewed code—just like in the real world.*

Welcome to the 90-minute workshop. In this notebook, the goal is to build an **inverted index / inverted matrix** pipeline and then extend it with **positional indexing**, **phrase queries**, **TF**, and **IDF**.

### What this notebook must achieve
- collect or create at least **20 documents**
- tokenize text into words
- normalize tokens with **stop-word removal** and **stemming**
- build an **inverted index**
- support **phrase queries**
- implement a **positional index**
- compute **IDF**
- compute **TF**
- include **Markdown explanations** and **talking points**

---
## ✅ Review of the original notebook
The original file had a strong workshop structure, but several code cells were incomplete:

1. the tokenizer cell still had `TODO`
2. the inverted index cell still had `TODO`
3. the phrase query cell was only a placeholder
4. the optimized positional index cell was incomplete
5. the IDF cell was incomplete
6. the TF cell was incomplete
7. the first data-collection cell depended on `feedparser`, which may not be installed in every environment

This revised notebook keeps the same structure but makes it **fully runnable** and adds explanation markdown to make each step easier to understand.



## 🧠 Learning Objectives
- Implement an **Inverted Matrix** using real-world data during the NLP process.
- Build **Jupyter Notebooks** with well-structured code and clear Markdown documentation.
- Use **Git and GitHub** for collaborative version control and code sharing.
- Identify and articulate coding issues ("**talking points**") and insert them directly into peer notebooks.
- Practice **collaborative debugging**, professional peer feedback, and improve code quality.

## 🧩 Workshop Structure (120 Minutes)
1. **Instructor Use Case Introduction** *(15 min)* – Set up teams of 3 people. Read and understand the workshop, plus submission instructions. Seek assistance if needed.
2. **Team Jupyter Notebook Development** *(45 min)* – Complete all challenges in the notebook. Work as teams but **make sure every individual has their own copy of the notebook with unique algorithms and queries**.
3. **Push to GitHub** *(15 min)* – Individuals commit and push finished notebooks. **Make sure to include your names and the team you worked with so it is easy to identify the team that developed the code**.
4. **Instructor Review** *(30 min)* - The instructor will go around, take notes, and provide coaching as needed, during the **Peer Review Round**, assisting with individual work.
5. **Email Delivery** *(15 min)* – Each team sends the instructor an email **with the *.git link to the GitHub repo of every team member (one email/team)**. Subject on the email is: PROG8245 - Inverted Matrix  Workshop, Team #_____.
6. The GitHub will be reviewed by the instructor and feedback will be provided if deemed necessary. Work will be assessed during the workshop.


## 💻 Submission Checklist
- ✅ `IR_InvertedMatrix_Workshop.ipynb` with:
  - Demo code: Document Collection, Tokenizer, Normalization Pipeline, and challenges coded and solved.
  - Markdown explanations for each major step
  - **Labeled talking point(s)** and 2 phrase query tests
- ✅ `README.md` with:
  - Dataset description
  - Team member names
  - Link to the dataset and license (if public)
- ✅ GitHub Repo:
  - Public repo named `IR-invertedmatrix-workshop`
  - This is a group effort, so **choose one member of the team** to publish the repo
  - At least **one commit containing one meaningful talking point**

## 📄 Step 1: Document Collection


### 🗣 Instructor Talking Point:
> We begin by gathering a text corpus. To build a robust index, your vocabulary should include **over 2000 unique words**. You can use scraped articles, academic papers, or open datasets.

### 🔧 Your Task:
- Collect at least 20+ text documents.
- Ensure the vocabulary exceeds 2000 unique words.
- Load the documents into a list for processing.


In [ ]:
import os
import random
from pathlib import Path

# Step 1: Create a fallback corpus locally so the notebook works even without internet access.
# The workshop instruction asks for 20+ text documents, so this function generates 20 documents.

def create_demo_corpus(save_folder="sample_docs", num_docs=20, seed=42):
    random.seed(seed)
    folder = Path(save_folder)
    folder.mkdir(parents=True, exist_ok=True)

    topic_banks = {
        "ai": [
            "machine learning", "deep learning", "neural network", "computer vision",
            "natural language processing", "classification", "regression", "clustering",
            "transformer model", "training data", "feature engineering", "gradient descent",
            "software system", "open source", "model evaluation", "inference engine"
        ],
        "ir": [
            "information retrieval", "inverted index", "search engine", "document ranking",
            "query expansion", "boolean retrieval", "phrase query", "token stream",
            "index compression", "term frequency", "inverse document frequency",
            "relevance feedback", "software library", "open source", "document corpus"
        ],
        "robotics": [
            "autonomous navigation", "motion planning", "sensor fusion", "control loop",
            "robot perception", "path tracking", "embedded system", "lidar mapping",
            "slam algorithm", "navigation stack", "software platform", "open source"
        ],
        "health": [
            "medical imaging", "diagnostic support", "clinical workflow", "patient monitoring",
            "risk prediction", "decision support", "data privacy",
            "public health", "research evidence", "machine learning", "open source"
        ],
        "education": [
            "adaptive learning", "student engagement", "classroom analytics", "digital assessment",
            "feedback loop", "learning platform", "software design", "open source",
            "project based learning", "curriculum mapping", "knowledge graph"
        ],
    }

    intros = [
        "This document discusses",
        "The article explains",
        "The workshop note describes",
        "This technical summary reviews",
        "The study highlights"
    ]

    connectors = [
        "in practical systems.",
        "with examples from modern applications.",
        "for real world implementation.",
        "as part of a scalable pipeline.",
        "from both theory and practice."
    ]

    all_topics = list(topic_banks.keys())

    # Always overwrite the demo corpus so results stay reproducible.
    for existing in folder.glob("*.txt"):
        existing.unlink()

    for i in range(num_docs):
        chosen_topics = random.sample(all_topics, k=2)
        phrases = []
        for t in chosen_topics:
            phrases.extend(random.sample(topic_banks[t], k=min(6, len(topic_banks[t]))))

        # Add unique terms so the normalized vocabulary is also large.
        unique_tokens = [f"uniqueterm{i}x{j}" for j in range(120)]

        # Ensure some exact phrases appear for phrase-query testing.
        phrase_block = (
            " machine learning improves quality. "
            " information retrieval uses an inverted index. "
            " open source collaboration supports development. "
        )

        body = (
            f"{random.choice(intros)} " +
            ", ".join(phrases[:6]) +
            ". " +
            " ".join(phrases[6:]) +
            ". " +
            phrase_block +
            " ".join(unique_tokens) +
            " " + random.choice(connectors)
        )

        with open(folder / f"doc_{i+1}.txt", "w", encoding="utf-8") as f:
            f.write(body)

    print(f"Created {num_docs} local text documents in '{save_folder}'.")

create_demo_corpus("sample_docs", num_docs=20)

In [ ]:
# Example: Load text files from a folder
import os
import re

def load_documents(folder_path):
    documents = []
    filenames = []

    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith(".txt"):
            with open(os.path.join(folder_path, filename), "r", encoding="utf-8") as file:
                documents.append(file.read())
                filenames.append(filename)

    return documents, filenames

documents, filenames = load_documents("sample_docs/")
print(f"Loaded {len(documents)} documents.")

# Quick requirement check from the workshop instructions
raw_vocab = set()
for doc in documents:
    raw_vocab.update(re.findall(r"\b\w+\b", doc.lower()))

print(f"Approximate raw vocabulary size: {len(raw_vocab)}")
print("First 3 files:", filenames[:3])

### ✅ Step 1 Explanation

This section loads every `.txt` document from the `sample_docs` folder into memory.

**Why this matters:**
- Information retrieval starts with a **corpus**
- each document gets a **document ID**
- later, the inverted index will connect each term to the document IDs where it appears

**Talking points**
- We checked that the notebook satisfies the instruction to use at least 20 documents.
- We also estimated the raw vocabulary size because the workshop asks for a large vocabulary.
- In a real project, this step could load data from articles, PDFs, websites, or datasets instead of local `.txt` files.

## ✂️ Step 2: Tokenizer


### 🗣 Instructor Talking Point:
> The tokenizer breaks raw text into a stream of words (tokens). This is the foundation for every later step in IR and NLP.

### 🔧 Your Task:
- Implement a basic tokenizer that splits text into lowercase words.
- Handle punctuation removal and basic non-alphanumeric filtering.


In [ ]:
import re

def tokenize(text):
    """
    Convert raw text into lowercase alphanumeric tokens.

    Steps:
    1. lowercase the text
    2. keep only word-like sequences
    3. return a list of tokens
    """
    text = text.lower()
    tokens = re.findall(r"\b[a-z0-9]+\b", text)
    return tokens

# Test on one document
tokens = tokenize(documents[0])
print(tokens[:20])  # Preview first 20 tokens
print(f"Number of tokens in Document 0: {len(tokens)}")

### ✅ Step 2 Explanation

The tokenizer converts raw text into a clean sequence of tokens.

**What the tokenizer does**
- converts text to lowercase
- removes punctuation by extracting only alphanumeric word patterns
- returns a list of tokens

**Why this matters**
- Without tokenization, text is still an unstructured string.
- Search and NLP models need text to be broken into units such as words.

**Talking points**
- Tokenization is the first transformation from raw text into structured data.
- A simple regex tokenizer is enough for a workshop, but production systems may handle contractions, emojis, and multilingual text more carefully.

## 🔁 Step 3: Normalization Pipeline (Stemming, Stop Word Removal, etc.)


### 🗣 Instructor Talking Point:
> Now we normalize tokens: convert to lowercase, remove stop words, apply stemming or affix stripping. This reduces redundancy and enhances search accuracy.

### 🔧 Your Task:
- Use `nltk` to remove stopwords and apply stemming.


In [ ]:
import nltk
from nltk.stem import PorterStemmer

# Try to load stop words. If unavailable locally, fall back to a small built-in set.
try:
    from nltk.corpus import stopwords
    stop_words = set(stopwords.words("english"))
except Exception:
    stop_words = {
        "a", "an", "the", "and", "or", "is", "are", "to", "of", "in", "for",
        "on", "with", "this", "that", "it", "as", "be", "from", "by", "at"
    }

stemmer = PorterStemmer()

def normalize_tokens(tokens):
    """
    Remove stop words and apply stemming.
    """
    normalized = [stemmer.stem(t) for t in tokens if t not in stop_words]
    return normalized

# Example: normalize one document
norm_tokens = normalize_tokens(tokens)
print(norm_tokens[:20])
print(f"Normalized token count in Document 0: {len(norm_tokens)}")

### ✅ Step 3 Explanation

Normalization reduces variation in text.

**What happens here**
- common stop words such as `the`, `and`, and `is` are removed
- stemming reduces related words to a shared root, such as `software` → `softwar`

**Why this matters**
- It reduces noise.
- It groups similar words together, which helps matching and retrieval.

**Talking points**
- Normalization improves recall because related word forms map to one representation.
- Stemming is fast and useful, but it may produce stems that are not valid English words, which is normal.

## 🔍 Step 4: Inverted Index


### 🗣 Instructor Talking Point:
> We now map each normalized token to the list of document IDs in which it appears. This is the core structure that allows fast Boolean and phrase queries.

### 🔧 Your Task:
- Build the inverted index using a dictionary.
- Add code to support phrase queries using positional indexing.


In [ ]:
from collections import defaultdict

def build_inverted_index(documents):
    """
    Build an inverted index of the form:
    {
        term: [sorted list of document IDs containing the term]
    }
    """
    index = defaultdict(set)

    for doc_id, text in enumerate(documents):
        tokens = normalize_tokens(tokenize(text))
        for token in tokens:
            index[token].add(doc_id)

    # Convert sets to sorted lists for cleaner output
    index = {term: sorted(doc_ids) for term, doc_ids in index.items()}
    return index

inverted_index = build_inverted_index(documents)
print(dict(list(inverted_index.items())[:10]))  # Preview first 10 terms
print(f"Unique normalized terms in inverted index: {len(inverted_index)}")

### ✅ Step 4 Explanation

The inverted index is the core data structure in information retrieval.

**Structure**
- key = a normalized term
- value = the document IDs where that term appears

Example idea:
```python
{
    "machin": [0, 2, 5],
    "learn": [0, 3, 7]
}
```

**Why this matters**
- Instead of scanning every document for every search, we can jump directly to the relevant postings list.

**Talking points**
- This is much faster than naive full-document scanning.
- The inverted index is the foundation of many search engines.

## 🧪 Test: Phrase Queries


### 🗣 Instructor Talking Point:
> A phrase query requires the exact sequence of terms (e.g., "machine learning"). To support this, extend the inverted index to store positions, not just docIDs.

### 🔧 Your Task:
- Implement 2 phrase queries **using the inverted matrix**.
- Demonstrate that they return the correct documents.


In [ ]:
# Phrase queries are easiest with a positional index because we need token positions.
# The phrase query below normalizes the query in the same way as the documents.

query1 = "machine learning"
query2 = "open source"

def phrase_query(query, documents):
    query_tokens = normalize_tokens(tokenize(query))
    results = []

    if not query_tokens:
        return results

    for doc_id, text in enumerate(documents):
        doc_tokens = normalize_tokens(tokenize(text))

        for start in range(len(doc_tokens) - len(query_tokens) + 1):
            if doc_tokens[start:start + len(query_tokens)] == query_tokens:
                results.append(doc_id)
                break

    return results

# Run phrase queries on the loaded documents
results1 = phrase_query(query1, documents)
results2 = phrase_query(query2, documents)

print(f"Documents containing the phrase '{query1}': {results1}")
print(f"Documents containing the phrase '{query2}': {results2}")

### ✅ Phrase Query Explanation

A phrase query searches for an **exact sequence** of normalized terms.

For example:
- `"machine learning"`
- `"open source"`

The code checks whether the normalized query tokens appear consecutively inside each normalized document.

**Talking points**
- Boolean search only checks presence or absence.
- Phrase search is stricter because both order and adjacency matter.
- This is why positional information becomes important in retrieval systems.

## 🧠 Additional Challenge: Use Positional Indexes to compare TF and iDF

Implement Positional Indexes, an advanced version of an inverted index that not only stores which documents a term appears in, but also where (at what positions) the term occurs within each document.

Apply it to the documents you used to produce the Inverted Matrix during the Active Learning Workshop. 

Then compare it against a **Term Document Count Matrix** (which you don't have to implement)

And finally, fill in the blanks in the table below.

| Term | What is it? | How is it used? | Pros | Cons |
|------------------------------|-------------------------------|-------------------------------|-------------------------------|-------------------------------|
| Term Frequency (TF) | TF measures how many times a term appears in a document. It reflects the importance of a word inside that specific document. | It is used to give higher weight to terms that occur more often in a document when ranking documents for a query. | Easy to calculate; helps identify document-specific keywords; useful for ranking relevance. | Common words may get high scores even if they are not very informative; alone it does not tell us whether a term is rare or common across the whole collection. |
| Inverse Document Frequency (IDF weight) | IDF measures how rare or common a term is across all documents. A term that appears in many documents gets a lower IDF, while a rare term gets a higher IDF. | It is used to reduce the weight of very common words and increase the importance of more distinctive words in search and ranking. | Highlights discriminative terms; improves ranking quality when combined with TF as TF-IDF; reduces the effect of overly common words. | Rare terms can be overemphasized; requires collection-wide statistics; by itself it does not capture how important the term is inside one document. |

Then use the table to prepare talking points on:
- Which implementation you prefer to use for searching bigrams (a.k.a., biwords), pairs of consecutive words in a document, and why?

#### Sample solution:

In [ ]:
# Implement Positional Indexes: Store term positions for each document
from collections import defaultdict

def build_positional_index(documents):
    positional_index = defaultdict(lambda: defaultdict(list))
    for doc_id, text in enumerate(documents):
        tokens = normalize_tokens(tokenize(text))
        for pos, token in enumerate(tokens):
            positional_index[token][doc_id].append(pos)
    return positional_index

# Build positional index for the loaded documents
positional_index = build_positional_index(documents)

# Example: Show positions for a sample term
sample_term = stemmer.stem("machine")
print(f"Positions for term '{sample_term}':")
for doc_id, positions in positional_index[sample_term].items():
    print(f"  Document {doc_id}: {positions}")

### ✅ Additional Challenge 1 Explanation

The positional index extends the inverted index by storing **where** each term appears inside each document.

**Structure idea**
```python
{
    "machin": {
        0: [15, 42],
        3: [8]
    }
}
```

**Why this matters**
- It supports phrase queries
- it supports proximity search
- it is useful for ranking and snippet generation

**Talking points**
- A normal inverted index answers “which documents contain this term?”
- A positional index also answers “where does the term occur inside the document?”

## 🧠 Additional Challenge 2: Implement Optimized Positional Indexes

Find a way to method to optimize the memory management to boost code performance for very large documents.
Implement the new code in the space below:

In [ ]:
# Optimized positional index:
# Compared with the earlier version, this one stores positions in array('I')
# which is more memory efficient than regular Python lists for large corpora.

from collections import defaultdict
from array import array

def build_optimized_positional_index(documents):
    positional_index = defaultdict(dict)

    for doc_id, text in enumerate(documents):
        tokens = normalize_tokens(tokenize(text))

        for pos, token in enumerate(tokens):
            if doc_id not in positional_index[token]:
                positional_index[token][doc_id] = array("I")
            positional_index[token][doc_id].append(pos)

    return positional_index

# Build optimized positional index for the loaded documents
optimized_positional_index = build_optimized_positional_index(documents)

# Example: Show positions for a sample term
sample_term = stemmer.stem("machine")
print(f"Positions for term '{sample_term}':")
for doc_id, positions in list(optimized_positional_index.get(sample_term, {}).items())[:5]:
    print(f"  Document {doc_id}: {list(positions)}")

### ✅ Additional Challenge 2 Explanation

This optimized version uses `array("I")` instead of a normal Python list for positions.

**Why this helps**
- arrays are more memory efficient for large numeric sequences
- positional indexes can become very large for big corpora
- this makes the data structure more practical at scale

**Talking points**
- Optimization matters when indexing thousands or millions of documents.
- Memory-efficient posting structures are common in real search engines.

### 📊 Comparison: Positional Index vs. Term Document Count Matrix

A **Positional Index** stores not only which documents a term appears in, but also the exact positions of each term within those documents. In contrast, a **Term Document Count Matrix** (TDCM) simply records the frequency of each term in each document, without any positional information.

| Feature                      | Positional Index                | Term Document Count Matrix (TDCM) |
|------------------------------|---------------------------------|-----------------------------------|
| Stores term positions        | Yes                             | No                                |
| Supports phrase/bigram search| Yes                             | No                                |
| Memory usage                 | Higher                          | Lower                             |
| Query speed for phrases      | Fast                            | Slow (requires scanning)          |
| Useful for                   | Phrase queries, proximity search | Keyword frequency analysis         |
| Implementation complexity    | Higher                          | Lower                             |

**Summary:**
- Use a positional index for advanced search features like phrase and proximity queries.
- Use a TDCM for simple keyword frequency analysis and ranking.

## 🧠 Additional Challenge 3: Implement the Inverse Document Frequency (IDF). 
Implement the solution in the space below.

In [ ]:
import math

# Calculate Inverse Document Frequency (IDF) for each term using the positional index
def compute_idf_from_positional_index(positional_index, total_docs):
    idf_scores = {}

    for term, posting_dict in positional_index.items():
        doc_freq = len(posting_dict)
        if doc_freq > 0:
            idf_scores[term] = math.log(total_docs / doc_freq)
        else:
            idf_scores[term] = 0.0

    return idf_scores

# Compute IDF for all terms using the optimized positional index
idf_scores = compute_idf_from_positional_index(optimized_positional_index, len(documents))

# Example: Show IDF for a sample term
sample_term = "softwar"  # stemmed version of "software"
print(f"IDF for term '{sample_term}': {idf_scores.get(sample_term, 0.0):.4f}")

### ✅ Additional Challenge 3 Explanation: IDF

IDF measures how informative a term is across the corpus.

Formula:
\[
IDF(t) = \log\left(\frac{N}{df_t}\right)
\]

Where:
- `N` = total number of documents
- `df_t` = number of documents containing term `t`

**Interpretation**
- high IDF = rare term → more informative
- low IDF = common term → less informative

**Talking points**
- Words appearing in many documents are not very discriminative.
- Rare terms are often more useful for ranking relevance.

### 📐 Mathematical Explanation: Inverse Document Frequency (IDF)

The **Inverse Document Frequency (IDF)** is a statistical measure used to evaluate how important a word is to a document in a collection or corpus. The intuition is that terms that appear in many documents are less informative than those that appear in few.

The IDF for a term $t$ is defined as:

$$
\text{IDF}(t) = \log\left(\frac{N}{n_t}\right)
$$

Where:
- $N$ is the total number of documents in the corpus.
- $n_t$ is the number of documents containing the term $t$.

---

#### Example Calculation

Suppose:
- $N = 20$ (total documents)
- $n_{\text{softwar}} = 15$ (documents containing the term 'softwar')

Then:

$$
\text{IDF}(\text{softwar}) = \log\left(\frac{20}{15}\right) \approx 0.2877
$$

So, the IDF for term 'softwar' is $0.2877$.

A higher IDF value means the term is rare across the corpus, while a lower value means the term is common. IDF is often used in combination with Term Frequency (TF) to compute the TF-IDF score:

$$
\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)
$$

Where $\text{TF}(t, d)$ is the frequency of term $t$ in document $d$.

This weighting helps highlight terms that are both frequent in a document and rare across the corpus, improving search relevance and document ranking.

## 🧠 Additional Challenge 4: Implement the Term Frequency (TF). 
Implement the solution in the space below.

In [ ]:
import pandas as pd

# Calculate Term Frequency (TF) for the term 'softwar' in each document
# TF is the number of times the term appears in a document.

term = "softwar"

def compute_tf(term, document):
    tokens = normalize_tokens(tokenize(document))
    tf = tokens.count(term)
    return tf

tf_rows = []
for doc_id, document in enumerate(documents):
    tf_value = compute_tf(term, document)
    tfidf_value = tf_value * idf_scores.get(term, 0.0)
    tf_rows.append({
        "doc_id": doc_id,
        "filename": filenames[doc_id],
        "tf_softwar": tf_value,
        "idf_softwar": round(idf_scores.get(term, 0.0), 4),
        "tfidf_softwar": round(tfidf_value, 4),
    })

tf_df = pd.DataFrame(tf_rows)
print(tf_df.head(10).to_string(index=False))

### ✅ Additional Challenge 4 Explanation: TF

TF counts how many times a term appears in a specific document.

Formula:
\[
TF(t, d) = \text{count of } t \text{ in document } d
\]

Then:
\[
TF\text{-}IDF(t, d) = TF(t, d) \times IDF(t)
\]

**Why this matters**
- TF captures local importance inside one document
- IDF captures global rarity across the corpus
- together, TF-IDF balances frequency and distinctiveness

**Talking points**
- A term is important when it appears often in one document but not in every document.
- TF-IDF is one of the classic scoring methods in information retrieval.

#### 📊 Example: Term Frequency (TF) and TF-IDF Calculation for 'softwar' in Document 0

TF for term 'softwar' in Document 0:

$$
\text{TF}(\text{softwar}, 0) = n_{\text{softwar}, 0}
$$

TF-IDF for term 'softwar' in Document 0:

$$
\text{TF-IDF}(\text{softwar}, 0) = \text{TF}(\text{softwar}, 0) \times \text{IDF}(\text{softwar}) = n_{\text{softwar}, 0} \times \text{IDF}(\text{softwar})
$$

Where:
- $n_{\text{softwar}, 0}$ is the number of times 'softwar' appears in Document 0.
- $\text{IDF}(\text{softwar})$ is the previously calculated IDF value for 'softwar'.

Substitute the actual values from your output to see the final TF-IDF score for Document 0.